## Service discovery & DNS

A service named `api` is reachable on its overlay network by the hostname **`api`** — but *how* that name resolves comes in two flavours, and picking right matters.

**Virtual IP (VIP) mode — the default.** `api` resolves to a single, stable **virtual IP**. The kernel (IPVS) load-balances incoming connections across all healthy tasks behind that one IP. Clients see one address and never need to know how many replicas exist or where they live; tasks coming and going is completely transparent.

**DNS round-robin (DNSRR) mode.** `api` resolves to the **IPs of all current tasks**, and each DNS lookup returns a different one. The client does its own load-balancing — and if a task dies, a client may get a stale IP and have to retry. You switch modes in the service definition:

```yaml
services:
  api:
    deploy:
      endpoint_mode: dnsrr    # or "vip", the default
```

**Use VIP for almost everything** — it's stable, kernel-fast, and hides replica churn. **Use DNSRR only** for clients that load-balance themselves and want the raw target IPs (some service meshes, custom gossip protocols).

One boundary worth internalising: Swarm's DNS resolves **only within the same overlay network**. A service on `frontend` *cannot* resolve a service that's only on `backend` — by design. That's the same segmentation-by-attachment idea from module 05, now the backbone of cluster security: put your public tier and your data tier on separate overlays, attach only what must talk to both, and the DNS boundary enforces the isolation for you. Service discovery isn't just convenience — it's how you structure who can reach whom across the cluster.